In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats


class DeGrootAgent:
    def __init__(self, n_agents, stubbornness=0.3, initial_opinions=None):
        self.n_agents = n_agents
        self.alpha = stubbornness
        self.agent_id = 0

        if initial_opinions is not None:
            self.opinions = np.array(initial_opinions)
        else:
            self.opinions = np.random.uniform(0, 1, n_agents)

        # Полный граф (без self‑loop)
        self.adjacency_matrix = np.ones((n_agents, n_agents))
        np.fill_diagonal(self.adjacency_matrix, 0)

        self.reset_bandit_stats()

    def reset_bandit_stats(self):
        self.n_pulls = np.zeros(self.n_agents)
        self.rewards = np.zeros(self.n_agents)
        self.mean_rewards = np.zeros(self.n_agents)
        self.ucb_values = np.zeros(self.n_agents)

    def get_reward(self, neighbor_id, new_opinion, old_opinion):
        """Награда = -|изменение мнения агента|"""
        return -abs(new_opinion - old_opinion)

    def epsilon_greedy(self, epsilon=0.1):
        if np.random.random() < epsilon:
            neighbor = np.random.choice([i for i in range(self.n_agents) if i != self.agent_id])
        else:
            valid_indices = [i for i in range(self.n_agents) if i != self.agent_id]
            neighbor = valid_indices[np.argmax(self.mean_rewards[valid_indices])]
        return neighbor

    def ucb(self, c=2.0):
        total_pulls = np.sum(self.n_pulls)
        for i in range(self.n_agents):
            if i == self.agent_id:
                self.ucb_values[i] = -np.inf
            elif self.n_pulls[i] == 0:
                self.ucb_values[i] = np.inf
            else:
                exploration = c * np.sqrt(np.log(total_pulls) / self.n_pulls[i])
                self.ucb_values[i] = self.mean_rewards[i] + exploration
        return np.argmax(self.ucb_values)

    def thompson_sampling(self):
        sampled_values = np.zeros(self.n_agents)
        for i in range(self.n_agents):
            if i == self.agent_id:
                sampled_values[i] = -np.inf
            elif self.n_pulls[i] == 0:
                sampled_values[i] = np.random.normal(0, 10)
            else:
                mean = self.mean_rewards[i]
                std = 1.0 / np.sqrt(self.n_pulls[i] + 1)
                sampled_values[i] = np.random.normal(mean, std)
        return np.argmax(sampled_values)

    def update_opinion(self, neighbor_id):
        old_opinion = self.opinions[self.agent_id]
        neighbor_mean = self.opinions[neighbor_id]
        new_opinion = self.alpha * old_opinion + (1 - self.alpha) * neighbor_mean
        self.opinions[self.agent_id] = new_opinion

        reward = self.get_reward(neighbor_id, new_opinion, old_opinion)
        self.n_pulls[neighbor_id] += 1
        self.rewards[neighbor_id] += reward
        self.mean_rewards[neighbor_id] = self.rewards[neighbor_id] / self.n_pulls[neighbor_id]

        return reward, new_opinion

    def run_simulation(self, strategy='epsilon_greedy', n_steps=100, **kwargs):
        history = {
            'opinions': [],
            'rewards': [],
        }

        history['opinions'].append(self.opinions.copy())

        for step in range(n_steps):
            # Выбор соседа
            if strategy == 'epsilon_greedy':
                epsilon = kwargs.get('epsilon', 0.2)
                neighbor = self.epsilon_greedy(epsilon)
            elif strategy == 'ucb':
                c = kwargs.get('c', 2.0)
                neighbor = self.ucb(c)
            elif strategy == 'thompson':
                neighbor = self.thompson_sampling()
            else:
                raise ValueError(f"Неизвестная стратегия: {strategy}")

            reward, new_opinion = self.update_opinion(neighbor)

            history['opinions'].append(self.opinions.copy())
            history['rewards'].append(reward)

        return history

def run_multiple_simulations_and_plot(
    strategies, n_agents=7, n_steps=100, n_runs=30, stubbornness=0.3
):


    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    colors = ["blue", "orange", "green"]
    labels = {
        "epsilon_greedy": r"$\varepsilon$-greedy",
        "ucb": "UCB",
        "thompson": "Thompson",
    }

    for idx, strategy in enumerate(strategies):

        cum_rewards_per_run = np.zeros((n_runs, n_steps + 1))
        opinions_per_run   = np.zeros((n_runs, n_steps + 1))

        for run in range(n_runs):

            np.random.seed(run)
            init_opinions = np.random.uniform(0, 1, n_agents)

            agent = DeGrootAgent(n_agents, stubbornness, init_opinions)
            hist = agent.run_simulation(strategy, n_steps)


            rewards = np.asarray(hist["rewards"])
            cum_rewards = np.concatenate([[0], np.cumsum(rewards)])
            cum_rewards_per_run[run, :] = cum_rewards

            opinions = [h[0] for h in hist["opinions"]]
            opinions_per_run[run, :] = opinions

        mean_cum = np.mean(cum_rewards_per_run, axis=0)
        se_cum   = np.std(cum_rewards_per_run, axis=0) / np.sqrt(n_runs)

        mean_op  = np.mean(opinions_per_run, axis=0)
        se_op    = np.std(opinions_per_run, axis=0) / np.sqrt(n_runs)


        ax1.plot(mean_cum, color=colors[idx], label=labels[strategy], linewidth=2, alpha=0.9)
        ax1.fill_between(
            range(n_steps + 1),
            mean_cum - se_cum,
            mean_cum + se_cum,
            color=colors[idx],
            alpha=0.3,
        )

        ax2.plot(mean_op, color=colors[idx], label=labels[strategy], linewidth=2, alpha=0.9)
        ax2.fill_between(
            range(n_steps + 1),
            mean_op - se_op,
            mean_op + se_op,
            color=colors[idx],
            alpha=0.3,
        )

    ax1.set_xlabel("Шаг")
    ax1.set_ylabel("Накопленная награда")

    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.set_xlabel("Шаг")
    ax2.set_ylabel("Среднее мнение")

    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()






strategies = ["epsilon_greedy", "ucb", "thompson"]

run_multiple_simulations_and_plot(
    strategies,
    n_agents=7,
    n_steps=100,
    n_runs=250,
    stubbornness=0.3
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

class DeGrootAgent:
    def __init__(self, n_agents, stubbornness=0.3, initial_opinions=None, true_opinion=0.5):
        self.n_agents = n_agents
        self.alpha = stubbornness
        self.agent_id = 0
        self.true_opinion = true_opinion

        if initial_opinions is not None:
            self.opinions = np.array(initial_opinions)
        else:
            self.opinions = np.random.uniform(0, 1, n_agents)

        # Полный граф (без self‑loop)
        self.adjacency_matrix = np.ones((n_agents, n_agents))
        np.fill_diagonal(self.adjacency_matrix, 0)

        self.reset_bandit_stats()

    def reset_bandit_stats(self):
        self.n_pulls = np.zeros(self.n_agents)
        self.rewards = np.zeros(self.n_agents)
        self.mean_rewards = np.zeros(self.n_agents)
        self.ucb_values = np.zeros(self.n_agents)

    def get_reward(self, neighbor_id, new_opinion, old_opinion):
        return -abs(new_opinion - self.true_opinion)

    def epsilon_greedy(self, epsilon=0.1):
        if np.random.random() < epsilon:
            neighbor = np.random.choice([i for i in range(self.n_agents) if i != self.agent_id])
        else:
            valid_indices = [i for i in range(self.n_agents) if i != self.agent_id]
            neighbor = valid_indices[np.argmax(self.mean_rewards[valid_indices])]
        return neighbor

    def ucb(self, c=2.0):
        total_pulls = np.sum(self.n_pulls)
        for i in range(self.n_agents):
            if i == self.agent_id:
                self.ucb_values[i] = -np.inf
            elif self.n_pulls[i] == 0:
                self.ucb_values[i] = np.inf
            else:
                exploration = c * np.sqrt(np.log(total_pulls) / self.n_pulls[i])
                self.ucb_values[i] = self.mean_rewards[i] + exploration
        return np.argmax(self.ucb_values)

    def thompson_sampling(self):
        sampled_values = np.zeros(self.n_agents)
        for i in range(self.n_agents):
            if i == self.agent_id:
                sampled_values[i] = -np.inf
            elif self.n_pulls[i] == 0:
                sampled_values[i] = np.random.normal(0, 10)
            else:
                mean = self.mean_rewards[i]
                std = 1.0 / np.sqrt(self.n_pulls[i] + 1)
                sampled_values[i] = np.random.normal(mean, std)
        return np.argmax(sampled_values)

    def update_opinion(self, neighbor_id):
        old_opinion = self.opinions[self.agent_id]
        neighbor_mean = self.opinions[neighbor_id]
        new_opinion = self.alpha * old_opinion + (1 - self.alpha) * neighbor_mean
        self.opinions[self.agent_id] = new_opinion

        reward = self.get_reward(neighbor_id, new_opinion, old_opinion)
        self.n_pulls[neighbor_id] += 1
        self.rewards[neighbor_id] += reward
        self.mean_rewards[neighbor_id] = self.rewards[neighbor_id] / self.n_pulls[neighbor_id]

        return reward, new_opinion

    def run_simulation(self, strategy='epsilon_greedy', n_steps=100, **kwargs):
        history = {
            'opinions': [],
            'rewards': [],
        }

        history['opinions'].append(self.opinions.copy())

        for step in range(n_steps):

            if strategy == 'epsilon_greedy':
                epsilon = kwargs.get('epsilon', 0.2)
                neighbor = self.epsilon_greedy(epsilon)
            elif strategy == 'ucb':
                c = kwargs.get('c', 2.0)
                neighbor = self.ucb(c)
            elif strategy == 'thompson':
                neighbor = self.thompson_sampling()
            else:
                raise ValueError(f"Неизвестная стратегия: {strategy}")

            reward, new_opinion = self.update_opinion(neighbor)

            history['opinions'].append(self.opinions.copy())
            history['rewards'].append(reward)

        return history

# Остальной код без изменений
def run_multiple_simulations_and_plot(
    strategies, n_agents=7, n_steps=100, n_runs=30, stubbornness=0.3, true_opinion=0.2
):


    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    colors = ["blue", "orange", "green"]
    labels = {
        "epsilon_greedy": r"$\varepsilon$-greedy",
        "ucb": "UCB",
        "thompson": "Thompson",
    }

    for idx, strategy in enumerate(strategies):

        cum_rewards_per_run = np.zeros((n_runs, n_steps + 1))
        opinions_per_run    = np.zeros((n_runs, n_steps + 1))

        for run in range(n_runs):

            np.random.seed(run)
            init_opinions = np.random.uniform(0, 1, n_agents)

            agent = DeGrootAgent(n_agents, stubbornness, init_opinions, true_opinion=true_opinion)
            hist = agent.run_simulation(strategy, n_steps)


            rewards = np.asarray(hist["rewards"])
            cum_rewards = np.concatenate([[0], np.cumsum(rewards)])  # 0 на шаге 0
            cum_rewards_per_run[run, :] = cum_rewards


            opinions = [h[0] for h in hist["opinions"]]
            opinions_per_run[run, :] = opinions


        mean_cum = np.mean(cum_rewards_per_run, axis=0)
        se_cum   = np.std(cum_rewards_per_run, axis=0) / np.sqrt(n_runs)

        mean_op  = np.mean(opinions_per_run, axis=0)
        se_op    = np.std(opinions_per_run, axis=0) / np.sqrt(n_runs)

        ax1.plot(mean_cum, color=colors[idx], label=labels[strategy], linewidth=2, alpha=0.9)
        ax1.fill_between(
            range(n_steps + 1),
            mean_cum - se_cum,
            mean_cum + se_cum,
            color=colors[idx],
            alpha=0.3,
        )


        ax2.plot(mean_op, color=colors[idx], label=labels[strategy], linewidth=2, alpha=0.9)
        ax2.axhline(y=true_opinion, color='red', linestyle='--', alpha=0.7)
        ax2.fill_between(
            range(n_steps + 1),
            mean_op - se_op,
            mean_op + se_op,
            color=colors[idx],
            alpha=0.3,
        )


    ax1.set_xlabel("Шаг")
    ax1.set_ylabel("Накопленная награда")

    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.set_xlabel("Шаг")
    ax2.set_ylabel("Среднее мнение")

    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

strategies = ["epsilon_greedy", "ucb", "thompson"]
run_multiple_simulations_and_plot(
    strategies,
    n_agents=10,
    n_steps=100,
    n_runs=200,
    stubbornness=0.3,
    true_opinion=0.1
)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from itertools import combinations

def run_multiple_simulations(n_runs=50, n_agents=10, n_steps=100, stubbornness=0.3, true_opinion=0.2):
    strategies = ["epsilon_greedy", "ucb", "thompson"]
    results = {
        s: {
            "rewards": [],
            "final_error": [],
            "diversity": [],
            "convergence_speed": [],
        }
        for s in strategies
    }

    for run in range(n_runs):
        print(f"\rПрогресс: {run+1}/{n_runs}", end="")

        np.random.seed(run * 42)
        initial_opinions = np.random.uniform(0, 1, n_agents)

        for strategy in strategies:
            agent = DeGrootAgent(n_agents, stubbornness, initial_opinions.copy(), true_opinion=true_opinion)

            if strategy == "epsilon_greedy":
                history = agent.run_simulation(strategy, n_steps, epsilon=0.1)
            elif strategy == "ucb":
                history = agent.run_simulation(strategy, n_steps, c=2.0)
            elif strategy == "thompson":
                history = agent.run_simulation(strategy, n_steps)


            rewards = np.asarray(history["rewards"])
            results[strategy]["rewards"].append(np.mean(rewards))


            final_opinions = np.asarray([h[0] for h in history["opinions"]])
            final_error = abs(final_opinions[-1] - true_opinion)
            results[strategy]["final_error"].append(final_error)

            unique_neighbors = np.sum(agent.n_pulls > 0) - 1
            results[strategy]["diversity"].append(unique_neighbors / (n_agents - 1))


            errors = np.abs(final_opinions - true_opinion)
            changes = np.diff(errors)
            threshold = 0.001
            conv_step = next((i for i, change in enumerate(changes) if change < threshold), n_steps - 1)
            results[strategy]["convergence_speed"].append(conv_step / n_steps)

    return results

def perform_statistical_tests(results):
    strategies = list(results.keys())
    metrics = ["rewards", "final_error", "diversity", "convergence_speed"]

    metric_names = {
        "rewards": "Средняя награда (-|мнение-истина|)",
        "final_error": "Финальная ошибка до истины",
        "diversity": "Разнообразие выбора соседей",
        "convergence_speed": "Скорость сходимости к истине",
    }

    statistical_tests = {}

    for metric in metrics:
        print(f"\n{'='*70}")
        print(f"Статистический анализ: {metric_names[metric]}")
        print('='*70)

        data = {s: np.asarray(results[s][metric]) for s in strategies}


        df_stats = pd.DataFrame({
            "Стратегия": strategies,
            "Среднее": [np.mean(data[s]) for s in strategies],
            "Стд.откл.": [np.std(data[s]) for s in strategies],
            "Медиана": [np.median(data[s]) for s in strategies],
            "Мин": [np.min(data[s]) for s in strategies],
            "Макс": [np.max(data[s]) for s in strategies],
        })
        print("\n1. Описательная статистика:")
        print(df_stats.to_string(index=False))

        print("\n2. Тест нормальности (Шапиро–Вилк):")
        for s in strategies:
            stat, p_val = stats.shapiro(data[s])
            status = "Нормальное" if p_val > 0.05 else "НЕ нормальное"
            print(f"{s:>15}: W={stat:.4f}, p={p_val:.4f} ({status})")


        stat_l, p_l = stats.levene(*[data[s] for s in strategies])
        homo = "Одинаковые" if p_l > 0.05 else "Разные"
        print(f"\n3. Левене (дисперсии): stat={stat_l:.4f}, p={p_l:.4f} ({homo})")

        print("\n4. Парные сравнения:")
        test_results = []
        for s1, s2 in combinations(strategies, 2):
            p_shap1, p_shap2 = stats.shapiro(data[s1])[1], stats.shapiro(data[s2])[1]
            use_t = p_shap1 > 0.05 and p_shap2 > 0.05

            if use_t and p_l > 0.05:
                stat, p_val = stats.ttest_ind(data[s1], data[s2])
                test_type = "T-тест"
            elif use_t:
                stat, p_val = stats.ttest_ind(data[s1], data[s2], equal_var=False)
                test_type = "Welch"
            else:
                stat, p_val = stats.mannwhitneyu(data[s1], data[s2])
                test_type = "Mann-Whitney U"

            significant = "★★★" if p_val < 0.01 else "★" if p_val < 0.05 else "-"

            mean_diff = np.mean(data[s1]) - np.mean(data[s2])
            if metric in ["final_error", "convergence_speed"]:
                better = s1 if mean_diff < 0 else s2
            else:
                better = s1 if mean_diff > 0 else s2

            test_results.append({
                "Сравнение": f"{s1} vs {s2}",
                "Тест": test_type,
                "Статистика": f"{stat:.4f}",
                "p-value": f"{p_val:.4f}",
                "Значимо": significant,
                "Лучшая": better,
            })

        df_tests = pd.DataFrame(test_results)
        print(df_tests.to_string(index=False))
        statistical_tests[metric] = df_tests

    return statistical_tests

def plot_statistical_comparison(results, statistical_tests):
    strategies = list(results.keys())
    metrics = ["rewards", "final_error", "diversity", "convergence_speed"]

    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle("Статистическое сравнение MAB стратегий (награда = -|мнение-истина|)", fontsize=16)

    ax = axes[0, 0]
    data_to_plot = [results[s][m] for s in strategies for m in metrics]
    labels = [f"{s}\n{metric}" for s in strategies for metric in ["Награда", "Ошибка", "Разнообр.", "Скорость"]]

    box = ax.boxplot(data_to_plot, labels=labels, patch_artist=True, showfliers=False)
    colors = plt.cm.Set3(np.linspace(0, 1, len(data_to_plot)))
    for patch, color in zip(box['boxes'], colors):
        patch.set_facecolor(color)
    ax.tick_params(axis='x', rotation=45)
    ax.set_ylabel("Значение метрики")
    ax.set_title("Распределение метрик")
    ax.grid(True, alpha=0.3)


    ax = axes[0, 1]
    means = [np.mean(results[s]["rewards"]) for s in strategies]
    errors = [stats.sem(results[s]["rewards"]) for s in strategies]

    bars = ax.bar(strategies, means, yerr=errors, capsize=5, alpha=0.8, edgecolor='black')
    ax.set_ylabel("Средняя награда")
    ax.set_title("Средние награды ± SEM")
    ax.grid(True, alpha=0.3)

    for bar, mean in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(errors)*0.1,
                f'{mean:.3f}', ha='center', va='bottom')


    ax = axes[1, 0]
    means_error = [np.mean(results[s]["final_error"]) for s in strategies]
    errors_error = [stats.sem(results[s]["final_error"]) for s in strategies]

    bars = ax.bar(strategies, means_error, yerr=errors_error, capsize=5, alpha=0.8, color='red', edgecolor='black')
    ax.set_ylabel("Финальная |ошибка|")
    ax.set_title("Точность приближения к истине")
    ax.grid(True, alpha=0.3)


    ax = axes[1, 1]
    means_div = [np.mean(results[s]["diversity"]) for s in strategies]
    bars = ax.bar(strategies, means_div, alpha=0.8, color='green', edgecolor='black')
    ax.set_ylabel("Доля уникальных соседей")
    ax.set_title("Разнообразие выбора")
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


results = run_multiple_simulations(n_runs=250, true_opinion=0.8)
stat_tests = perform_statistical_tests(results)
plot_statistical_comparison(results, stat_tests)